In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

print("Libraries loaded!")

Libraries loaded!


In [2]:
model = joblib.load(
    "../models/best_model.pkl"
)

print("Model loaded!")
import urllib
from sqlalchemy import create_engine

server = r"DEVIL\SQLEXPRESS"
database = "ecommerce"

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + params
)

df = pd.read_sql(
    """
    SELECT *
    FROM dbo.Customer_Churn_ML
    """,
    engine
)

print(df.shape)

Model loaded!


c:\Users\sahuj\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


(55906, 7)


In [4]:
prediction_data = df.copy()

customer_ids = prediction_data[
    "customer_unique_id"
]

X = prediction_data.drop(
    columns=[
        "customer_unique_id",
        "churn_target"
    ],
    errors="ignore"
)

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(0)

prediction_data["churn_probability"] = (
    model.predict_proba(X)[:, 1]
)

prediction_data["predicted_churn"] = (
    model.predict(X)
)

prediction_data[
    [
        "customer_unique_id",
        "churn_probability",
        "predicted_churn"
    ]
].head()

,customer_unique_id,churn_probability,predicted_churn
0,0000f46a3911fa3c0805444483337064,0.861023,1
1,0000f6ccb0745a6a4b88665a16c9f078,0.698316,1
2,0004aac84e0df4da2b147fca70cf8255,0.594941,1
3,00053a61a98854899e70ed204dd4bafe,0.351228,0
4,0005e1862207bf6ccc02e4228effd9a0,0.778159,1


In [5]:
high_risk = (
    prediction_data
    .sort_values(
        "churn_probability",
        ascending=False
    )
    .head(20)
)

high_risk

,customer_unique_id,recency,frequency,monetary,avg_order_value,customer_lifetime_days,churn_target,churn_probability,predicted_churn
12929,3b0610a6b6c37bf5e849e71914f238c6,367,1,1246.339966,1246.339966,0,1,0.967848,1
701,0326524848da9311350236471586bd61,366,1,1826.069946,1826.069946,0,1,0.967848,1
19834,5a7881e38b4a49918a70658e1dd42a09,367,1,1936.270020,1936.270020,0,1,0.967848,1
2765,0cb6bcb4572e64a3394ede6448db0482,367,1,1228.729980,1228.729980,0,1,0.967848,1
26203,77dc1d016a6ff92c1eb364ecb3cc6aae,363,1,1409.119995,1409.119995,0,1,0.967848,1
23414,6b2af0e98d36916cbc15761bf6048220,364,1,1534.579956,1534.579956,0,1,0.967848,1
7861,23eeaf77f8c29ea22ced2a3fa0bc360b,364,1,1155.560059,1155.560059,0,1,0.967848,1
5582,198dc221b76657718f8c38d49daa83c9,365,1,1284.250000,1284.250000,0,1,0.967848,1
36428,a6b61ba2ddd9a18d0256d3396c5d5f02,369,1,1231.290039,1231.290039,0,1,0.967848,1
55849,ffba9f9dff87b05e310ecc46c8591044,368,1,1626.829956,1626.829956,0,1,0.967848,1


In [6]:
import os

os.makedirs(
    "../outputs",
    exist_ok=True
)

prediction_data.to_csv(
    "../outputs/customer_churn_predictions.csv",
    index=False
)

print(
    "Predictions saved successfully!"
)

Predictions saved successfully!
